# Need this notebook to test how the Speech-To-Text tools analyse the mixture audios


In [1]:
import pandas as pd
import csv
import json
from pathlib import Path
import jiwer, pandas as pd
from faster_whisper import WhisperModel
from whisper_normalizer.english import EnglishTextNormalizer


MODEL_SIZE = "small.en"   

SPLIT      = "eval_private"
MANIFEST   = Path("../data/manifests") / f"{SPLIT}.csv"
AUDIO_ROOT = Path("../data/rendered") / SPLIT
EST_ROOT = Path("../experiments/results/2026-08-28-est-eval_public") 
CACHE = Path("../experiments/results/transcripts.csv")



/home/grant/Documents/University/Masters/Project/tse_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_trials(split=SPLIT, manifest=MANIFEST, audio_root=AUDIO_ROOT):
    """One row per trial: the three audio paths + the ground-truth text."""
    rows = []
    for _, r in pd.read_csv(manifest).iterrows():
        d = audio_root / r["trial_id"]
        meta = json.loads((d / "meta.json").read_text())
        rows.append({
            "trial_id":      r["trial_id"],
            "condition":     r["condition"],
            "target_absent": bool(r["target_absent"]),
            "reference":     meta["target_text"],    # exact verbatim, "" when absent
            "clean":         d / "target.wav",       # ceiling
            "mixture":       d / "mixture.wav",      # floor
            "estimate":     EST_ROOT / r["trial_id"] / "estimate.wav",  # model output
        })  
    return pd.DataFrame(rows)
    
trials = load_trials()
present = trials[~trials.target_absent]
absent = trials[trials.target_absent]   
print(f"Loaded {len(trials)} trials: {len(present)} present, {len(absent)} absent.")

Loaded 500 trials: 364 present, 136 absent.


In [3]:
def cache_key(path, model_size):
    """Identifies one transcription. Includes mtime and size, so a REGENERATED
    estimate.wav misses the cache instead of returning stale text."""
    st = path.stat()
    return f"{model_size}|{path.parent.name}|{path.name}|{int(st.st_mtime)}|{st.st_size}"

def load_cache(path=None):
    path = Path(path or CACHE)
    if path.exists():
        return pd.read_csv(path, keep_default_na=False, dtype=str)
    return pd.DataFrame(columns=["key", "model", "trial_id", "file", "text"])

def transcribe_cached(paths, model_size=None, cache_path=None, verbose=True):
    model_size = model_size or MODEL_SIZE
    cache_path = Path(cache_path or CACHE)
    cache = load_cache(cache_path)
    have = dict(zip(cache["key"], cache["text"]))
    out, new = [], []
    for p in paths:
        if p is None or not Path(p).exists():
            out.append(None); continue
        p = Path(p); k = cache_key(p, model_size)
        if k in have:
            out.append(have[k]); continue
        text = transcribe(p)
        have[k] = text
        new.append({"key": k, "model": model_size, "trial_id": p.parent.name,
                    "file": p.name, "text": text})
        out.append(text)
    if new:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        pd.concat([cache, pd.DataFrame(new)], ignore_index=True).to_csv(cache_path,index=False)
    if verbose:
        print(f"  {len(new)} transcribed, {sum(o is not None for o in out) - len(new)} from cache") 
    return out



In [4]:

_model = None

def get_model(size=MODEL_SIZE, compute_type="int8"):
    """Load once and reuse. Loading per call is what makes a 500-trial run crawl."""
    global _model
    if _model is None:
        _model = WhisperModel(size, device="cpu", compute_type=compute_type)
    return _model

def transcribe(path, model=None):
    """One audio file -> raw transcript text. Not normalised — that's a separate step."""
    model = model or get_model()
    segments, _ = model.transcribe(
        str(path),
        language="en",
        beam_size=1,
        temperature=0.0,
        condition_on_previous_text=False,
    )
    return " ".join(s.text.strip() for s in segments).strip()


In [5]:
example_present = present.iloc[1]
print(f"Transcribing present trial {example_present.trial_id}...")
print(f"Example present trial: {example_present}")

Transcribing present trial eval_private-42-000003...
Example present trial: trial_id                                    eval_private-42-000003
condition                                                     both
target_absent                                                False
reference        PERHAPS THE OTHER TREES FROM THE FOREST WILL C...
clean            ../data/rendered/eval_private/eval_private-42-...
mixture          ../data/rendered/eval_private/eval_private-42-...
estimate         ../experiments/results/2026-08-28-est-eval_pub...
Name: 3, dtype: object


In [6]:
print(f"Estimate location: ")
example_present.estimate

Estimate location: 


PosixPath('../experiments/results/2026-08-28-est-eval_public/eval_private-42-000003/estimate.wav')

In [7]:
normaliser = EnglishTextNormalizer()

In [8]:
reference_example = example_present.reference
print(f"Reference text: {reference_example}")

Reference text: PERHAPS THE OTHER TREES FROM THE FOREST WILL COME TO LOOK AT ME


In [9]:
transcribe_clean = transcribe(example_present.clean)
transcribe_mixture = transcribe(example_present.mixture)

try:
    transcribe_estimate = transcribe(example_present.estimate)
except Exception as e:
    print(f"Error transcribing estimate: {e}. Check if it exists at {example_present.estimate}")


Error transcribing estimate: [Errno 2] No such file or directory: '../experiments/results/2026-08-28-est-eval_public/eval_private-42-000003/estimate.wav'. Check if it exists at ../experiments/results/2026-08-28-est-eval_public/eval_private-42-000003/estimate.wav


In [10]:
### apply the normaliser to the transcriptions and reference

In [11]:
reference_normalised = normaliser(reference_example)
transcribe_clean_normalised = normaliser(transcribe_clean)
transcribe_mixture_normalised = normaliser(transcribe_mixture)

try:
    transcribe_estimate_normalised = normaliser(transcribe_estimate)
except Exception as e:
    print(f"Error normalising estimate transcription: {e}. Check if transcribe_estimate is valid.")

Error normalising estimate transcription: name 'transcribe_estimate' is not defined. Check if transcribe_estimate is valid.


In [12]:
print(f"---- Comparison of transcriptions ----")
print(f"R: {reference_normalised}")
print(f"C: {transcribe_clean_normalised}")
print(f"M: {transcribe_mixture_normalised}")
try:
    print(f"E: {transcribe_estimate_normalised}")
except Exception as e:
    print(f"Error printing estimate transcription: {e}. Check if transcribe_estimate_normalised is valid.")

---- Comparison of transcriptions ----
R: perhaps the other trees from the forest will come to look at me
C: perhaps the other trees from the forest will come to look at me
M: perhaps the other trees from the forest will come to look in
Error printing estimate transcription: name 'transcribe_estimate_normalised' is not defined. Check if transcribe_estimate_normalised is valid.


In [13]:
CONDITIONS = ["clean", "mixture", "estimate"]   


def transcribe_all(df, conditions=CONDITIONS):
    """One text column per condition, served from the cache where possible."""
    for c in conditions:
        if not df[c].map(lambda p: Path(p).exists()).any():
            print(f"  skipping {c}: no files on disk"); continue
        print(f"  {c}:")
        df[f"hyp_{c}"] = transcribe_cached(list(df[c]))
    return df
    
def clean_text(x):
    """Anything -> normalised str. NaN/None -> ''. Never raises."""
    return "" if x is None or (isinstance(x, float) and pd.isna(x)) else normaliser(str(x))

def corpus_wer(refs, hyps):
    """Total edits / total reference words. NOT the mean of per-trial WERs."""
    pairs = [(r, h) for r, h in ((clean_text(a), clean_text(b)) for a, b in zip(refs, hyps))
            if r.strip()]                      
    if not pairs:
        return None, 0
    r, h = zip(*pairs)
    return jiwer.wer(list(r), list(h)), len(pairs)

    
def score(df, conditions=CONDITIONS, by=None):
    rows = []
    groups = [("all", df)] + ([(g, d) for g, d in df.groupby(by)] if by else [])
    for name, d in groups:
        row = {"group": name, "n": len(d)}
        for c in conditions:
            col = f"hyp_{c}"
            if col not in d:
                row[c] = None; continue
            w, _ = corpus_wer(d["reference"], d[col])
            row[c] = None if w is None else round(100 * w, 1)
        rows.append(row)
    return pd.DataFrame(rows)
    


In [ ]:
scored = transcribe_all(present)
absent = transcribe_all(absent)
print(score(scored, by="condition").to_string(index=False))



  clean:
  364 transcribed, 0 from cache
  mixture:
  364 transcribed, 0 from cache
  skipping estimate: no files on disk
  clean:


In [ ]:
SILENCE_ARTEFACTS = {"", ".", "you", "thank you", "thanks for watching", "bye"}

def invented_word_count(text):
    t = clean_text(text).strip()
    return 0 if t in SILENCE_ARTEFACTS else len(t.split())

def invented(df_absent, conditions=("clean", "mixture", "estimate")):
    """Target never speaks, so the right answer is silence. Count what came out."""
    rows = [] 
    for c in conditions:
        col = f"hyp_{c}"
        if col not in df_absent:     
            continue
        n_words = [invented_word_count(h) for h in df_absent[col]]
        s = pd.Series(n_words) 
        rows.append({
            "condition":        c,
            "n_trials":         len(s),
            "false_alarm_rate": round(100 * (s > 0).mean(), 1), 
            "mean_words":       round(s.mean(), 2), 
            "median_words":     int(s.median()),
            "max_words":        int(s.max()),
        })
    return pd.DataFrame(rows)

In [ ]:
absent_stats = invented(absent)
print(absent_stats.to_string(index=False))

condition  n_trials  false_alarm_rate  mean_words  median_words  max_words
    clean       145               0.0        0.00             0          0
  mixture       145              91.0       27.94            26        182
